# Merge & Push PolyLLM Chairman Model

This notebook loads the LoRA adapter (`sameetvipat/polyllm-chairman`) via unsloth's built-in adapter loading, then pushes the full merged fp16 weights back to HuggingFace using `push_to_hub_merged`.

**Required:** GPU runtime (Kaggle T4 x2 or P100)

**Time:** ~15 minutes

**Result:** `sameetvipat/polyllm-chairman` will contain full merged fp16 model weights and work via the HuggingFace Inference API.

## Step 1 — Install dependencies

In [ ]:
!pip install -q unsloth peft transformers accelerate huggingface_hub

## Step 2 — Config

In [ ]:
HF_TOKEN     = "hf_YOUR_WRITE_TOKEN"                        # HuggingFace write token
ADAPTER_REPO = "sameetvipat/polyllm-chairman"               # adapter / target repo
BASE_MODEL   = "mistralai/Mistral-7B-Instruct-v0.3"         # full fp16 base (NOT the bnb-4bit variant)
SAVE_DIR     = "/kaggle/working/merged_chairman"

print(f"Adapter:    {ADAPTER_REPO}")
print(f"Base model: {BASE_MODEL}")

## Step 3 — Load base model in fp16 and apply adapter

Using the full `mistralai/Mistral-7B-Instruct-v0.3` (fp16, ~14GB) instead of the bnb-4bit variant. This avoids all quantization issues: `merge_and_unload()` is clean, saved weights are proper fp16 that the HF Inference API can load without bitsandbytes.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Load full fp16 base model (~14GB, fits in T4 16GB VRAM)
print("Loading fp16 base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype = torch.float16,
    device_map  = "auto",
    token       = HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
print("✅ Base model loaded in fp16")

# Apply LoRA adapter
model = PeftModel.from_pretrained(model, ADAPTER_REPO, token=HF_TOKEN)
print("✅ LoRA adapter applied")

# Merge — safe since base is fp16 (no quantization rounding errors)
model = model.merge_and_unload()
print("✅ Adapter merged into base weights")

## Step 4 — Save fp16 weights and upload

In [ ]:
import os, transformers.modeling_utils as _mu
from huggingface_hub import HfApi

# Patch transformers 5.5 bug (broken revert_weight_conversion for Mistral)
_mu.revert_weight_conversion = lambda m, sd: sd

# Save merged fp16 model to disk
print("Saving fp16 model to disk...")
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ Saved to {SAVE_DIR}")

# Verify size
total = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, files in os.walk(SAVE_DIR) for f in files
)
print(f"   Total size: {total / 1e9:.1f} GB  (expect ~14 GB)")

# Upload to HuggingFace
print(f"\nUploading to {ADAPTER_REPO} ...")
api = HfApi()
api.upload_folder(
    folder_path    = SAVE_DIR,
    repo_id        = ADAPTER_REPO,
    token          = HF_TOKEN,
    commit_message = "Replace with merged fp16 weights",
)
print(f"✅ Done! https://huggingface.co/{ADAPTER_REPO}")

## Step 5 — Set pipeline_tag on the model card

The HF Serverless Inference API requires `pipeline_tag: text-generation` in the model card. `save_pretrained_merged` doesn't write a README, so we set it explicitly.

In [ ]:
from huggingface_hub import ModelCard

try:
    card = ModelCard.load(ADAPTER_REPO, token=HF_TOKEN)
except Exception:
    card = ModelCard("---\npipeline_tag: text-generation\n---\n\n# PolyLLM Chairman\n")

card.data.pipeline_tag = "text-generation"
card.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
print("✅ pipeline_tag: text-generation set on model card")

## Step 6 — Verify the HF Inference API responds

In [ ]:
import requests
from huggingface_hub import HfApi

# ── 1. Check what's actually in the repo ─────────────────────────────────────
api = HfApi()
print("Repo contents:")
for info in api.list_repo_tree(ADAPTER_REPO, token=HF_TOKEN):
    size = getattr(info, "size", 0) or 0
    print(f"  {info.path:45s}  {size / 1e9:.2f} GB" if size > 1e6 else f"  {info.path}")

# ── 2. Try old inference endpoint as a comparison ────────────────────────────
print("\n--- New router (router.huggingface.co) ---")
r1 = requests.post(
    f"https://router.huggingface.co/hf-inference/models/{ADAPTER_REPO}",
    headers={"Authorization": f"Bearer {HF_TOKEN}"},
    json={"inputs": "Say hello.", "parameters": {"max_new_tokens": 10, "return_full_text": False}},
)
print("Status:", r1.status_code, r1.text[:200])

print("\n--- Old endpoint (api-inference.huggingface.co) ---")
r2 = requests.post(
    f"https://api-inference.huggingface.co/models/{ADAPTER_REPO}",
    headers={"Authorization": f"Bearer {HF_TOKEN}"},
    json={"inputs": "Say hello.", "parameters": {"max_new_tokens": 10}},
)
print("Status:", r2.status_code, r2.text[:200])

---
Once Step 6 shows `✅ Inference API working!`, restart your PolyLLM backend and Stage 3B will use the fine-tuned chairman model.